# Fig. S7E: Expression changes of representative genes for each GSEA term shown in Fig. S7D

Prepare representative-gene expression and membership tables for `FigS7E_plot.R`.
Requires `selected_term2gene.tsv`, a separately curated term–gene table.


In [1]:
import os
import pandas as pd
from collections import defaultdict

In [2]:
# Define paths
basedir = "/path/to/project"
directory_out = "FigS7E"

# Output directory
path_out = f"{basedir}/output/revise_final/{directory_out}/"

# Curated term–gene table supplied separately
gene_table_path = f"{basedir}/output/260818_revise/rev_FigS6B_two_projects/chatgpt/selected_term2gene.tsv"
# GSEA and DESeq2 results
input_gsea_path = f"{basedir}/output/revise_final/FigS7D/gsea/"
input_dseq_path = f"{basedir}/output/revise_final/FigS7D/deseq2/"

os.makedirs(path_out, exist_ok=True)
print("saving files to:", path_out)


saving files to: /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/FigS7E/


# Select genes

In [3]:
# Pool leading-edge membership across comparisons
ttg = defaultdict(list)

for dirpath, dirnames, filenames in os.walk(input_gsea_path):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # Read the GSEA results
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]
        print("\nprocessing...", project)
        print("size:", res.shape)
        print("#NaN:", res.isna().sum().sum())
        print("#uniq Term:", len(set(res["Term"].to_list())))
        
        # Select terms
        selected_terms = [
            "Interferon Alpha Response",
            "IL-6/JAK/STAT3 Signaling",
            "WIELAND_UP_BY_HBV_INFECTION",
            "Xenobiotic Metabolism",
            "Adipogenesis",
        ]

        # Collect unique leading-edge genes for each selected term
        for idx in res.index:
            term = res.loc[idx, "Term"]
            if term not in selected_terms:
                continue
            print(term)
            genes = res.loc[idx, "Lead_genes"].split(";")
            if len(genes) != len(set(genes)):
                print("genes not unique")
            for gene in genes:
                if gene not in ttg[term]:
                    ttg[term].append(gene)


processing... PRJNA774885
size: (51, 10)
#NaN: 0
#uniq Term: 51
Interferon Alpha Response
IL-6/JAK/STAT3 Signaling
WIELAND_UP_BY_HBV_INFECTION
Xenobiotic Metabolism
Adipogenesis

processing... PRJEB63475
size: (51, 10)
#NaN: 0
#uniq Term: 51
Xenobiotic Metabolism
Adipogenesis
WIELAND_UP_BY_HBV_INFECTION
IL-6/JAK/STAT3 Signaling
Interferon Alpha Response


In [4]:
# Create the term–gene table
df = pd.DataFrame(
    [(k, v) for k, vs in ttg.items() for v in vs],
    columns=["term", "gene"]
)

df.to_csv(path_out + "term2gene.tsv" ,sep="\t")
# Preview results
print(df.shape)
print(df.isna().sum().sum())
df.head(3)

(287, 2)
0


,term,gene
0,Interferon Alpha Response,OGFR
1,Interferon Alpha Response,EPSTI1
2,Interferon Alpha Response,PARP9


# Create heatmap tables

In [5]:
# Preview the selected gene table
genedf = pd.read_table(gene_table_path)
genedf.head(3)

,term,gene
0,Interferon Alpha Response,RSAD2
1,Interferon Alpha Response,EIF2AK2
2,Interferon Alpha Response,IRF7


In [6]:
# Check selected genes against the leading-edge genes
# Pool leading-edge membership across comparisons
test_dic = defaultdict(set)
all_genes = set()

# Read the selected gene table
genedf = pd.read_table(gene_table_path)
genes_oi = []
for g in genedf["gene"].to_list():
    if g not in genes_oi:
        genes_oi.append(g)
print("# uniq gene oi:", len(set(genes_oi)))
print("size of gene df:", genedf.shape)
t_order = []
for t in genedf["term"].to_list():
    if t not in t_order:
        t_order.append(t)

for dirpath, dirnames, filenames in os.walk(input_gsea_path):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # Read the GSEA results
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]

        # Keep the selected terms
        for idx in res.index:
            term = res.loc[idx, "Term"]
            if term not in selected_terms:
                continue
            genes = res.loc[idx, "Lead_genes"].split(";")

            # Collect leading-edge genes
            for gene in genes:
                all_genes.add(gene)
                # Record term membership for selected genes
                if gene not in genes_oi:
                    continue
                test_dic[gene].add(term)

# Check that selected genes occur in the GSEA results
print(f"\n#all genes: {len(all_genes)}")
for goi in genes_oi:
    if goi not in sorted(list(all_genes)):
        print(f"### {goi} not exist! ###")

# Create gene–term associations
testdf = pd.DataFrame(
    [(k, v) for k, vs in test_dic.items() for v in sorted(list(vs))],
    columns=["gene", "term"]
)
print(test_dic)

# Create the gene–term membership matrix
term_bin = (pd.crosstab(testdf["gene"], testdf["term"]) > 0).astype(int)
term_bin = term_bin[t_order]
term_bin.columns = term_bin.columns.map(lambda x: x.replace("-", "_"))
# Order genes for the heatmap
term_bin = term_bin.sort_values(
            "gene",
            key=lambda s: pd.Categorical(s, categories=genes_oi, ordered=True)
        )
# Save tables
term_bin.to_csv(path_out + "term_anno.csv")

# Preview results
term_bin.head(3)

# uniq gene oi: 27
size of gene df: (28, 2)

#all genes: 266
defaultdict(<class 'set'>, {'IFIH1': {'Interferon Alpha Response'}, 'IRF1': {'Interferon Alpha Response', 'IL-6/JAK/STAT3 Signaling'}, 'EIF2AK2': {'Interferon Alpha Response'}, 'B2M': {'Interferon Alpha Response', 'WIELAND_UP_BY_HBV_INFECTION'}, 'TRIM25': {'Interferon Alpha Response'}, 'STAT2': {'Interferon Alpha Response', 'IL-6/JAK/STAT3 Signaling'}, 'USP18': {'Interferon Alpha Response'}, 'RSAD2': {'Interferon Alpha Response'}, 'CD74': {'Interferon Alpha Response', 'WIELAND_UP_BY_HBV_INFECTION'}, 'IRF7': {'Interferon Alpha Response'}, 'SOCS1': {'IL-6/JAK/STAT3 Signaling'}, 'STAT1': {'IL-6/JAK/STAT3 Signaling', 'WIELAND_UP_BY_HBV_INFECTION'}, 'SOCS3': {'IL-6/JAK/STAT3 Signaling'}, 'IL6ST': {'IL-6/JAK/STAT3 Signaling'}, 'STAT3': {'IL-6/JAK/STAT3 Signaling'}, 'FGL2': {'WIELAND_UP_BY_HBV_INFECTION'}, 'CIDEA': {'Adipogenesis'}, 'NQO1': {'Xenobiotic Metabolism'}, 'ALDH2': {'Xenobiotic Metabolism', 'Adipogenesis'}, 'CD36': {'Xeno

term,Interferon Alpha Response,IL_6/JAK/STAT3 Signaling,WIELAND_UP_BY_HBV_INFECTION,Xenobiotic Metabolism,Adipogenesis
gene,,,,,
RSAD2,1,0,0,0,0
EIF2AK2,1,0,0,0,0
IRF7,1,0,0,0,0


In [7]:
# Create the gene-by-project fold-change matrix
dfs = []
for dirpath, dirnames, filenames in os.walk(input_dseq_path):
    for fname in filenames:
        if not(fname.startswith("result") and (fname.endswith("_hs_gallus_and_Gene.txt"))):
            continue

        # Read the DESeq2 results
        path = os.path.join(dirpath, fname)
        project = pd.read_table(path)
        proname = fname.split("_")[1]
        print("\nprocessing...", proname)

        # Keep the selected genes
        df_gene = project.loc[project["gene_name_hs"].isin(genes_oi), ["gene_name_hs", "log2FoldChange"]].reset_index(drop=True).copy()
        df_gene["Project"] = proname
        print(f"size for {proname}: {df_gene.shape}")

        # Collect results for this project
        dfs.append(df_gene)

# Combine results across projects
df_cat = pd.concat(dfs)
print("\nsize for all:", df_cat.shape)
print(df_cat.head(3))
# Pivot to a gene-by-project matrix
df_pivot = (df_cat.pivot_table(index="gene_name_hs", columns="Project", values="log2FoldChange").sort_index())
print("\nsize for df pivot:", df_pivot.shape)
print("#NaN for df pivot:", df_pivot.isna().sum().sum())
# Apply the selected gene order
df_pivor_sorted = df_pivot.sort_values(
            "gene_name_hs",
            key=lambda s: pd.Categorical(s, categories=genes_oi, ordered=True)
        )

print("size for sorted:", df_pivor_sorted.shape)
# Save tables
df_pivor_sorted.to_csv(path_out + "gene_project_log2FC_matrix.csv")
# Preview results
df_pivor_sorted.head(3)


processing... PRJNA774885
size for PRJNA774885: (27, 3)

processing... PRJEB63475
size for PRJEB63475: (27, 3)

size for all: (54, 3)
  gene_name_hs  log2FoldChange      Project
0         CD36        0.057701  PRJNA774885
1         FGL2        0.938451  PRJNA774885
2        HMOX1        0.038990  PRJNA774885

size for df pivot: (27, 2)
#NaN for df pivot: 0
size for sorted: (27, 2)


Project,PRJEB63475,PRJNA774885
gene_name_hs,,
RSAD2,-0.067344,3.238871
EIF2AK2,0.631668,1.913080
IRF7,2.080364,0.781303
